In [106]:
import numpy as np
import pandas as pd
import kagglehub

dataset_name = "eward96/harry-potter-and-the-philosophers-stone-script"
path_raw = "data/raw/"
path_input = path_raw + "hp_script.csv"
path_processed = "data/processed/"
path_output_train_data = path_processed + "train_data.txt"
path_output_train_labels = path_processed + "train_labels.txt"
path_output_test_data = path_processed + "test_data.txt"
path_output_test_labels = path_processed + "test_labels.txt"

DOWNLOAD_DATASET = False

In [107]:
if DOWNLOAD_DATASET:
    path = kagglehub.dataset_download(dataset_name, output_dir=path_raw)
    print("Path to dataset files:", path)

# 1. EDA

In [108]:
df = pd.read_csv(path_input, encoding="latin")
df = df.drop(columns=["ID_number", "scene"])
df.head()

,character_name,dialogue
0,Albus Dumbledore,"I should have known that you would be here, Pr..."
1,Minerva McGonagall,"Good evening, Professor Dumbledore. Are the ru..."
2,Albus Dumbledore,"I'm afraid so, Professor. The good, and the bad."
3,Minerva McGonagall,And the boy?
4,Albus Dumbledore,Hagrid is bringing him.


## 1.1. Null values

Check that there are no null values :

In [109]:
df.isna().sum()

character_name    0
dialogue          0
dtype: int64

## 1.2. Data cleaning

A special character (, or \x92, or \u0092) is found in some lines of the raw CSV file. I see that there are 4 lines with this character :

In [110]:
for _, row in df[df["dialogue"].str.contains("\x92")].iterrows():
    print(row["dialogue"])

Oh, um, I'd appreciate if you didnt tell anyone at Hogwarts about that. Strictly speaking, I'm not allowed to do magic.
Sorry, but whats curious?
One of a wizard's most rudimentary skills is levitation, the ability to make objects fly. Uh, do you all have your feathers? Good. Now, uh, don't forget the nice wrist movement weve been practicing, hmm? The swish and flick. Everyone.
Honestly, dont you two read? The Philosopher's Stone is a legendary substance with astonishing powers. It will turn any metal into pure gold and produces the Elixir of Life, which will make the drinker immortal.


And I replace this character with an apostrophe, as it seems to be the right character for all the cases :

In [111]:
df["dialogue"] = df["dialogue"].str.replace("\x92", "'")

Later in this notebook, I observe some wrongly read special character (, or \x85, or \u0085). I see that there are several lines with this sequence :

In [112]:
n_print = 10
for _, row in df[df["dialogue"].str.contains("\x85")].iterrows():
    print(row["dialogue"])
    n_print -= 1
    if n_print > 0:
        continue
    else:
        break

Albus, do you really think it's safe, leaving him with these people? I've watched them all day. They're the worst sort of Muggles imaginable. They really are
Can you hear me? It's just I've never talked to a snake before. Do you I mean do you talk to people often?
No, you've made a mistake. I can't be a-a wizard. I mean, I'm just Harry. Just Harry.
All students must be equipped with one standard size two pewter cauldron, and may bring if they desire, either an owl, a cat or a toad. Could we find all this in London?
I still need a wand.
And who owned that wand?
Oh, we do not speak his name. The wand chooses the wizard, Mr. Potter. It's not always clear why, but I think it is clear that we can expect great things from you. After all, He-Who-Must-Not-Be-Named did great things terrible, yes, but great.
First, and understand this, Harry, 'cause it's very important, not all wizards are good. Some of them go bad. A few years ago, there was one wizard who went as bad as you can go. 

I'd think that this special character is used in the script to represent pauses in the speech, or suspensive dots "...".

In [113]:
# Some of these \x85 are already followed by a space, so I start replacing these, then I replace the rest with an extra space.
df["dialogue"] = df["dialogue"].str.replace("\x85 ", "... ")
df["dialogue"] = df["dialogue"].str.replace("\x85", "... ")

In [114]:
for _, row in df[df["dialogue"].str.contains("Anything but Slytherin")].iterrows():
    print(row["dialogue"])

Not Slytherin... Anything but Slytherin.


### 1.3. Top 4 most talkative characters

I display the characters who have most lines in the movie :

In [115]:
df["character_name"].value_counts()[:10]

character_name
Harry Potter          230
Ron Weasley           120
Hermione Granger       92
Rubeus Hagrid          81
Minerva McGonagall     31
Albus Dumbledore       24
Vernon Dursley         23
Dudley Dursley         17
Quirinus Quirrell      17
Neville Longbottom     14
Name: count, dtype: int64

# 2. Dataset creation

I choose the top 4 most talkative characters, and I cap them to have only 80 lines in my dataset (50 for training, 30 for testing). In the end, I shuffle the training data.

In [116]:
chosen_characters = ["Harry Potter", "Ron Weasley", "Hermione Granger", "Rubeus Hagrid"]
n_dialogue = 80

train_size = 50
test_size = n_dialogue - train_size

train_data = np.array([], dtype=object)
train_labels = np.array([], dtype=int)

test_data = np.array([], dtype=object)
test_labels = np.array([], dtype=int)

for character_idx, character in enumerate(chosen_characters):

    df_subset = df[df["character_name"] == character]
    dialogue = df_subset.sample(n=n_dialogue)["dialogue"].values

    train_data = np.concatenate([train_data, dialogue[:train_size]])
    train_labels = np.concatenate([train_labels, [character_idx] * train_size])

    test_data = np.concatenate([test_data, dialogue[train_size:]])
    test_labels = np.concatenate([test_labels, [character_idx] * test_size])

# Shuffle

train_perm = np.random.permutation(len(train_data))
train_data = train_data[train_perm]
train_labels = train_labels[train_perm]

# Save

np.savetxt(path_output_train_data, train_data, fmt="%s")
np.savetxt(path_output_train_labels, train_labels, fmt="%s")
np.savetxt(path_output_test_data, test_data, fmt="%s")
np.savetxt(path_output_test_labels, test_labels, fmt="%s")